# Assemble MSD serology data for the Sound Life cohort

We have two types of serology data derived from MSD (Meso Scale Diagnostics) assays:

**IgG Serology**: Measurement of the total concentration of IgG antibodies in plasma that are able to bind to specific flu HA antigens. A standard curve of calibration samples are used to convert signal values to concentrations.

**Hemagglutination Inhibition (HAI Assay)**: Measurement of the percent inhibition of HA binding to labeled RBC vessicles. Serum/Plasma-free blank samples are used as a reference, and the fraction of this reference signal that is observed for each plasma treatment sample are used to compute the percent of inhibition. A 1:4 dilution series of reference standard samples are also included, but are not used as a direct standard curve.

In [1]:
from datetime import date

import hisepy
import numpy as np
import os
import pandas as pd
import polars as pl
import re

import plotly.express as px

In [2]:
if not os.path.isdir('output'):
    os.mkdir('output')

In [3]:
out_files = []

### Helper functions

In [4]:
def specimens_to_kits(specimens):
    sample_kits = []
    for specimen in specimens:
        if 'PL' in specimen:
            sample_kit = re.sub('PL([0-9]+)-.+','KT\\1', specimen)
            sample_kits.append(sample_kit)
        else:
            sample_kits.append(None)

    return sample_kits

In [5]:
def element_id(n = 3):
    import periodictable
    from random import randrange
    rand_el = []
    for i in range(n):
        el = randrange(0,118)
        rand_el.append(periodictable.elements[el].name)
    rand_str = '-'.join(rand_el)
    return rand_str

## Load metadata

### MSD Assay Metadata

In [6]:
assay_meta_uuid = '5e3a7f65-d92a-4af0-8983-1d31a9000e14'
assay_meta_csv = hisepy.cache_files([assay_meta_uuid])[0]
assay_meta = pl.read_csv(assay_meta_csv)

In [7]:
assay_meta.head()

msd.assayName,msd.assayType,msd.antigenProtein,msd.antigenName,msd.antigenFullName,msd.antigenVirus,msd.antigenStrain,msd.antigenSubtype,msd.antigenIsolate
str,str,str,str,str,str,str,str,str
"""Flu A/Brisbane (H1N1)""","""IgG Serology""","""HA""","""A/Brisbane""","""A/Brisbane/02/2018 (H1N1) pdm0…","""Influenza""","""A""","""H1N1""","""Brisbane"""
"""Flu A/Hong Kong (H3N2)""","""IgG Serology""","""HA""","""A/Hong Kong""","""A/Hong Kong/2671/2019 (H3N2)-l…","""Influenza""","""A""","""H3N2""","""Hong Kong"""
"""Flu A/Michigan (H1N1)""","""IgG Serology""","""HA""","""A/Michigan""","""A/Michigan/45/2015 (H1N1)-like…","""Influenza""","""A""","""H1N1""","""Michigan"""
"""Flu A/Victoria (H1N1)""","""IgG Serology""","""HA""","""A/Victoria""","""A/Victoria/2570/2019 (H1N1) pd…","""Influenza""","""A""","""H1N1""","""Victoria"""
"""Flu B/Colorado HA""","""IgG Serology""","""HA""","""B/Colorado""","""B/Colorado/06/2017-like virus …","""Influenza""","""B""","""Victoria""","""Colorado"""


In [8]:
assay_meta = assay_meta.select(
    ['msd.assayName', 'msd.antigenName', 'msd.antigenSubtype', 'msd.antigenFullName']
)

In [9]:
assay_meta.head()

msd.assayName,msd.antigenName,msd.antigenSubtype,msd.antigenFullName
str,str,str,str
"""Flu A/Brisbane (H1N1)""","""A/Brisbane""","""H1N1""","""A/Brisbane/02/2018 (H1N1) pdm0…"
"""Flu A/Hong Kong (H3N2)""","""A/Hong Kong""","""H3N2""","""A/Hong Kong/2671/2019 (H3N2)-l…"
"""Flu A/Michigan (H1N1)""","""A/Michigan""","""H1N1""","""A/Michigan/45/2015 (H1N1)-like…"
"""Flu A/Victoria (H1N1)""","""A/Victoria""","""H1N1""","""A/Victoria/2570/2019 (H1N1) pd…"
"""Flu B/Colorado HA""","""B/Colorado""","""Victoria""","""B/Colorado/06/2017-like virus …"


### Sample Metadata

In [10]:
meta_uuid = 'af25e3e7-25c1-4476-afb4-926bd201db8f'

In [11]:
meta_file = hisepy.cache_files([meta_uuid])[0]

In [12]:
meta = pl.read_csv(meta_file)

In [13]:
meta.shape

(868, 19)

Add flu year, then drop the drawDate field for drawYear

In [14]:
meta = meta.with_columns(
    pl.when(pl.col('sample.drawDate') < '2020-07')
    .then(pl.lit('2019-2020'))
    .when(pl.col('sample.drawDate') < '2021-07')
    .then(pl.lit('2020-2021'))
    .otherwise(pl.lit('2021-2022'))
    .alias('vaccine.year')
).with_columns(
    pl.col('sample.drawDate').str.replace('-.+','')
).rename({'sample.drawDate':'sample.drawYear'})

In [15]:
baselines = meta.filter(
    pl.col('sample.visitName').str.contains('Flu Year')
).filter(
    pl.col('sample.visitName').str.contains('Day 0')
)

In [16]:
baselines.shape

(176, 20)

In [17]:
meta.head()

,cohort.cohortGuid,subject.subjectGuid,subject.biologicalSex,subject.cmv,subject.bmi,subject.race,subject.ethnicity,subject.birthYear,subject.ageAtFirstDraw,subject.covidVaxDose1.daysSinceFirstVisit,subject.covidVaxDose2.daysSinceFirstVisit,sample.sampleKitGuid,sample.visitName,sample.drawYear,sample.subjectAgeAtDraw,sample.daysSinceFirstVisit,specimen.specimenGuid,pipeline.fileGuid,vaccine.year
i64,str,str,str,str,f64,str,str,i64,i64,f64,f64,str,str,str,i64,i64,str,str,str
0,"""BR1""","""BR1001""","""Female""","""Negative""",23.0,"""Caucasian""","""Non-Hispanic origin""",1987,32,null,null,"""KT00001""","""Flu Year 1 Day 0""","""2019""",32,0,"""PB00001-01""","""fec489f9-9a74-4635-aa91-d2bf09…","""2019-2020"""
1,"""BR1""","""BR1002""","""Male""","""Negative""",22.0,"""Caucasian""","""Non-Hispanic origin""",1991,28,440.0,461.0,"""KT00002""","""Flu Year 1 Day 0""","""2019""",28,0,"""PB00002-01""","""7c0c7979-eebd-4aba-b5b2-6e76b4…","""2019-2020"""
2,"""BR1""","""BR1003""","""Female""","""Negative""",21.0,"""Caucasian""","""Non-Hispanic origin""",1989,30,440.0,461.0,"""KT00003""","""Flu Year 1 Day 0""","""2019""",30,0,"""PB00003-01""","""40efd03a-cb2f-4677-af42-a056cb…","""2019-2020"""
3,"""BR1""","""BR1004""","""Male""","""Negative""",22.0,"""Caucasian""","""Non-Hispanic origin""",1989,30,543.0,563.0,"""KT00004""","""Flu Year 1 Day 0""","""2019""",30,0,"""PB00004-01""","""68fbcd34-1d63-461d-8195-df5b8d…","""2019-2020"""
4,"""BR1""","""BR1005""","""Female""","""Negative""",20.0,"""Caucasian""","""Non-Hispanic origin""",1992,27,451.0,492.0,"""KT00006""","""Flu Year 1 Day 0""","""2019""",27,0,"""PB00006-01""","""ea8d98e9-e99e-4dc6-9e78-9866e0…","""2019-2020"""


In [18]:
age_groups = {
    'BR1': 'Young Adult',
    'BR2': 'Older Adult'
}

In [19]:
meta = meta.with_columns(
    pl.Series(
        name = 'subject.ageGroup',
        values = [age_groups[c] for c in meta['cohort.cohortGuid']]
    )
)

In [20]:
meta.columns

['',
 'cohort.cohortGuid',
 'subject.subjectGuid',
 'subject.biologicalSex',
 'subject.cmv',
 'subject.bmi',
 'subject.race',
 'subject.ethnicity',
 'subject.birthYear',
 'subject.ageAtFirstDraw',
 'subject.covidVaxDose1.daysSinceFirstVisit',
 'subject.covidVaxDose2.daysSinceFirstVisit',
 'sample.sampleKitGuid',
 'sample.visitName',
 'sample.drawYear',
 'sample.subjectAgeAtDraw',
 'sample.daysSinceFirstVisit',
 'specimen.specimenGuid',
 'pipeline.fileGuid',
 'vaccine.year',
 'subject.ageGroup']

In [21]:
keep_meta_cols = [
    'cohort.cohortGuid',
    'subject.subjectGuid',
    'sample.sampleKitGuid',
    'subject.biologicalSex',
    'subject.birthYear',
    'subject.ageAtFirstDraw',
    'subject.ageGroup',
    'subject.race',
    'subject.ethnicity',
    'subject.cmv',
    'sample.visitName',
    'sample.drawYear',
    'sample.subjectAgeAtDraw',
    'sample.daysSinceFirstVisit',
    'vaccine.year'
]

In [22]:
meta = meta.select(keep_meta_cols)

In [23]:
meta.head()

cohort.cohortGuid,subject.subjectGuid,sample.sampleKitGuid,subject.biologicalSex,subject.birthYear,subject.ageAtFirstDraw,subject.ageGroup,subject.race,subject.ethnicity,subject.cmv,sample.visitName,sample.drawYear,sample.subjectAgeAtDraw,sample.daysSinceFirstVisit,vaccine.year
str,str,str,str,i64,i64,str,str,str,str,str,str,i64,i64,str
"""BR1""","""BR1001""","""KT00001""","""Female""",1987,32,"""Young Adult""","""Caucasian""","""Non-Hispanic origin""","""Negative""","""Flu Year 1 Day 0""","""2019""",32,0,"""2019-2020"""
"""BR1""","""BR1002""","""KT00002""","""Male""",1991,28,"""Young Adult""","""Caucasian""","""Non-Hispanic origin""","""Negative""","""Flu Year 1 Day 0""","""2019""",28,0,"""2019-2020"""
"""BR1""","""BR1003""","""KT00003""","""Female""",1989,30,"""Young Adult""","""Caucasian""","""Non-Hispanic origin""","""Negative""","""Flu Year 1 Day 0""","""2019""",30,0,"""2019-2020"""
"""BR1""","""BR1004""","""KT00004""","""Male""",1989,30,"""Young Adult""","""Caucasian""","""Non-Hispanic origin""","""Negative""","""Flu Year 1 Day 0""","""2019""",30,0,"""2019-2020"""
"""BR1""","""BR1005""","""KT00006""","""Female""",1992,27,"""Young Adult""","""Caucasian""","""Non-Hispanic origin""","""Negative""","""Flu Year 1 Day 0""","""2019""",27,0,"""2019-2020"""


### Serology standard ids
We'll need to convert the sample IDs provided for serology standards (controls) to their source samples (sample.sampleKitGuid) so that they can be grouped appropriately for analysis.

A table with these sample ID -> sampleKitGuid associations were compiled from our LIMS database, then were stored in HISE via watchfolder.

In [24]:
std_uuid = '878ed657-7c57-4b6b-8d2b-da68c243da88'
std_csv = hisepy.cache_files([std_uuid])[0]
std_ids = pl.read_csv(std_csv)

In [25]:
std_ids = std_ids.rename({'specimen.specimenGuid': 'Sample'})

## IgG Serology Data

### Read Serology from HISE

In [26]:
msd_uuid = 'df68e382-a14e-4d86-b5c8-25963c084a54'

In [27]:
all_serology = pl.read_csv(hisepy.cache_files([msd_uuid])[0], infer_schema_length = 10000)
all_serology.shape

(46383, 22)

### Prepare Serology data
Standardize IDs, filter for relevant data, and standardize column names

Convert specimen IDs to sampleKitGuid for matching to sample metadata

In [28]:
all_serology = all_serology.with_columns(
    pl.Series(name = 'sample.sampleKitGuid',
              values = specimens_to_kits(all_serology['Sample']))
)

Clean up batch IDs

In [29]:
all_serology = all_serology.with_columns(
    pl.col('Batch ID').str.replace(',.+', '')
)

Add a plate ID (one plate per batch in this case)

In [30]:
all_serology = all_serology.with_columns(
    pl.Series(
        name = 'Plate ID',
        values = [b + '_P1' for b in all_serology['Batch ID']]
    )
)

Filter for custom panel runs

In [31]:
sample_serology = all_serology.filter(
    pl.col('Notes').str.contains('Custom PLAN-00072')
)
sample_serology.shape

(27720, 24)

Select samples in sample metadata set

In [32]:
sample_serology = sample_serology.filter(
    pl.col('sample.sampleKitGuid').is_in(meta['sample.sampleKitGuid'].unique().to_list())
)
sample_serology.shape

(13272, 24)

Select controls and standards from batches that match our samples

In [33]:
control_serology = all_serology.filter(
    # Keep selected batches
    pl.col('Batch ID').is_in(sample_serology['Batch ID'].unique().to_list())
).filter(
    # Drop samples
    pl.col('sample.sampleKitGuid').is_null()
)
control_serology.shape

(8372, 24)

Add sample names as sample.sampleKitGuid for controls and QC standards

In [34]:
control_serology = control_serology.drop(
    'sample.sampleKitGuid'
).join(
    std_ids,
    how = 'left',
    on = 'Sample'
).select(sample_serology.columns)

Combine samples and controls

In [35]:
serology = pl.concat([sample_serology, control_serology])
serology.shape

(21644, 24)

Is there any difference between Signal and Adjusted Signal?

In [36]:
serology = serology.with_columns(
    (abs(pl.col('Adjusted Signal') - pl.col('Signal'))).alias('diff')
)

In [37]:
max(serology['diff'])

0.0

No difference, so let's use Signal as the primary raw value.

In [38]:
keep_cols = {
    'sample.sampleKitGuid': 'sample.sampleKitGuid',
    'Sample': 'specimen.specimenGuid', 
    'Batch ID': 'msd.batchID',
    'Plate ID': 'msd.plateID',
    'well': 'msd.wellID', 
    'Assay': 'msd.assayName',
    'Dilution': 'msd.sampleDilution', 
    'Concentration': 'msd.standardConc',
    'Signal': 'msd.signalWell', 
    'Mean': 'msd.signalMean', 
    'CV': 'msd.signalPercentCV',
    'Calc. Concentration': 'msd.concWell', 
    'Calc. Conc. Mean': 'msd.concMean', 
    'Calc. Conc. CV': 'msd.concPercentCV'
}

In [39]:
serology = serology.select(keep_cols.keys()).rename(keep_cols)
serology.shape

(21644, 14)

In [40]:
serology = serology.with_columns(
    pl.col('msd.standardConc').cast(pl.Float64).round(6),
    pl.col('msd.concWell').cast(pl.Float64),
    pl.col('msd.concMean').cast(pl.Float64)
)

In [41]:
serology.head()

sample.sampleKitGuid,specimen.specimenGuid,msd.batchID,msd.plateID,msd.wellID,msd.assayName,msd.sampleDilution,msd.standardConc,msd.signalWell,msd.signalMean,msd.signalPercentCV,msd.concWell,msd.concMean,msd.concPercentCV
str,str,str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64
"""KT00001""","""PL00001-03""","""MSD-00029""","""MSD-00029_P1""","""C09""","""Flu A/Victoria (H1N1)""",10000.0,null,4422.0,4713.0,10.632476,1787.528284,1907.391212,10.809815
"""KT00001""","""PL00001-03""","""MSD-00029""","""MSD-00029_P1""","""C08""","""Flu A/Victoria (H1N1)""",10000.0,null,5292.0,4713.0,10.632476,2145.471789,1907.391212,10.809815
"""KT00001""","""PL00001-03""","""MSD-00029""","""MSD-00029_P1""","""C07""","""Flu A/Victoria (H1N1)""",10000.0,null,4426.0,4713.0,10.632476,1789.173564,1907.391212,10.809815
"""KT00001""","""PL00001-03""","""MSD-00029""","""MSD-00029_P1""","""C09""","""Flu A/Hong Kong (H3N2)""",10000.0,null,12026.0,12162.0,1.038986,6827.241774,6910.345676,1.1202
"""KT00001""","""PL00001-03""","""MSD-00029""","""MSD-00029_P1""","""C08""","""Flu A/Hong Kong (H3N2)""",10000.0,null,12183.0,12162.0,1.038986,6923.392038,6910.345676,1.1202


### Join assay and sample metadata

In [42]:
serology = serology.join(
    meta,
    how = 'left',
    on = 'sample.sampleKitGuid'
).join(
    assay_meta,
    how = 'left',
    on = 'msd.assayName'
)

In [43]:
serology.shape

(21644, 31)

In [44]:
serology_order = [
    'cohort.cohortGuid',
    'subject.subjectGuid', 'subject.birthYear', 'subject.ageAtFirstDraw', 'subject.ageGroup', 
    'subject.biologicalSex', 'subject.race', 'subject.ethnicity', 'subject.cmv',
    'sample.sampleKitGuid', 'sample.visitName', 'sample.drawYear', 'sample.subjectAgeAtDraw',
    'sample.daysSinceFirstVisit', 'vaccine.year',
    'specimen.specimenGuid',
    'msd.batchID', 'msd.plateID', 'msd.wellID',
    'msd.antigenName', 'msd.antigenSubtype', 'msd.antigenFullName',
    'msd.sampleDilution', 'msd.standardConc',
    'msd.signalWell', 'msd.signalMean', 'msd.signalPercentCV', 
    'msd.concWell', 'msd.concMean', 'msd.concPercentCV'
]

In [45]:
serology = serology.select(serology_order)
serology.shape

(21644, 30)

### Separate duplicate measures

Some samples have been measured more than once across multiple batches. We'll separate out these duplicate measures, as they can be confusing for downstream analysis. 

**Single measures** retain the most recent measurement in these cases by sorting on batch and plate IDs (which are alphanumerically ordered).

**Duplicate measures** retain all measurements of the duplicated samples, including the most recent, to enable analysis of cross-batch replication.

In [46]:
treat_serology = serology.filter(
    pl.col('specimen.specimenGuid').str.contains('PL')
)
control_serology = serology.filter(
    ~pl.col('specimen.specimenGuid').str.contains('PL')
)

In [47]:
last_serology = treat_serology.select(
    ['sample.sampleKitGuid','specimen.specimenGuid', 'msd.batchID']
).unique(    
).sort(
    ['sample.sampleKitGuid', 'specimen.specimenGuid', 'msd.batchID']
).group_by('sample.sampleKitGuid').last()

In [48]:
last_serology.shape

(523, 3)

In [49]:
single_serology = treat_serology.filter(
    pl.col('specimen.specimenGuid').is_in(last_serology['specimen.specimenGuid'].to_list())
)

duplicate_kits = treat_serology.filter(
    ~pl.col('specimen.specimenGuid').is_in(last_serology['specimen.specimenGuid'].to_list())
)['sample.sampleKitGuid'].unique().to_list()
duplicate_serology = treat_serology.filter(
    ~pl.col('sample.sampleKitGuid').is_in(duplicate_kits)
)

In [50]:
single_serology['sample.sampleKitGuid'].unique().len()

523

In [51]:
len(duplicate_kits)

71

In [52]:
single_control_serology = control_serology.filter(
    pl.col('msd.batchID').is_in(single_serology['msd.batchID'].unique().to_list())
)
duplicate_control_serology = control_serology.filter(
    pl.col('msd.batchID').is_in(duplicate_serology['msd.batchID'].unique().to_list())
)

In [53]:
single_serology = pl.concat(
    [single_serology, single_control_serology]
)
duplicate_serology = pl.concat(
    [duplicate_serology, duplicate_control_serology]
)

In [54]:
single_serology.shape

(18235, 30)

In [55]:
duplicate_serology.shape

(14672, 30)

### Assemble sample-level data

For sample-level results, we'll exclude controls to provide the simplest data to end users looking for on-study results

In [56]:
single_sample_serology = single_serology.drop(
    ['msd.wellID', 'msd.signalWell', 'msd.concWell']
).unique().filter(
    pl.col('specimen.specimenGuid').str.contains('PL')
)
single_sample_serology.shape

(3667, 27)

In [57]:
duplicate_sample_serology = duplicate_serology.drop(
    ['msd.wellID', 'msd.signalWell', 'msd.concWell']
).unique().filter(
    pl.col('specimen.specimenGuid').str.contains('PL')
)
duplicate_sample_serology.shape

(3170, 27)

### Output serology results

In [58]:
single_serology_out = 'output/sound-life_flu_serology_single_raw_{d}.csv'.format(d = date.today())
single_serology.write_csv(single_serology_out)
out_files.append(single_serology_out)

In [59]:
duplicate_serology_out = 'output/sound-life_flu_serology_duplicates_raw_{d}.csv'.format(d = date.today())
duplicate_serology.write_csv(duplicate_serology_out)
out_files.append(duplicate_serology_out)

In [60]:
single_sample_serology_out = 'output/sound-life_flu_serology_single_{d}.csv'.format(d = date.today())
single_sample_serology.write_csv(single_sample_serology_out)
out_files.append(single_sample_serology_out)

In [61]:
duplicate_sample_serology_out = 'output/sound-life_flu_serology_duplicates_{d}.csv'.format(d = date.today())
duplicate_sample_serology.write_csv(duplicate_sample_serology_out)
out_files.append(duplicate_sample_serology_out)

## HAI Assays

### Additional HAI metadata

#### Calibration/control positions
Calibration samples were placed in the first two columns (in duplicate) in a decreasing 1:4 dilution series from the top row (A) to the bottom (F or G). The final rows of these columns (G and H or just H) are blanks, which serve as the positive control for HA binding. For the first two pilot experiments (EXP-01042 and EXP-01072), a 7-point dilution series was used with one blank row. For all others, a 6-point dilution curve was used with two blank rows.

In [62]:
control_rows = ['A','B','C','D','E','F','G','H']
control_cols = ['01','02']

control_dils = [1,4,16,64,256,1024,0,0]
pilot_dils = [1,4,16,64,256,1024,4096,0]

blank_str = 'Diluent Only Control (Blank)'

control_samples = {}
pilot_samples = {}
control_specimens = {}
pilot_specimens = {}
control_dilutions = {}
pilot_dilutions = {}

for i in range(len(control_rows)):
    row = control_rows[i]
    control_dil = control_dils[i]
    pilot_dil = pilot_dils[i]
    
    for col in control_cols:
        rc = row + col
        control_dilutions[rc] = control_dil
        pilot_dilutions[rc] = pilot_dil

        if row == 'G':
            control_samples[rc] = blank_str
            control_specimens[rc] = blank_str
            
            pilot_samples[rc] = 'HAI Reference Standard'
            pilot_specimens[rc] = 'HAI Reference Standard 1:' + str(pilot_dil)
        if row in ['H']:
            control_samples[rc] = blank_str
            control_specimens[rc] = blank_str

            pilot_samples[rc] = blank_str
            pilot_specimens[rc] = blank_str
        else:
            sample = 'HAI Reference Standard'
            if row == 'A':
                specimen = 'HAI Reference Standard 1X'
            else:
                specimen = 'HAI Reference Standard 1:' + str(control_dil)

            control_samples[rc] = sample
            control_specimens[rc] = specimen
            pilot_samples[rc] = sample
            pilot_specimens[rc] = specimen

#### Pilot sample metadata adaptation
Sample metadata for our pilot experiment (EXP-01042) - not present in the data table, but can be added based on the well positions.

In [63]:
pilot_sample_info_uuid = '93837e91-cd3a-48cc-9cab-d59f0054b7cd'
pilot_sample_info_xlsx = hisepy.cache_files([pilot_sample_info_uuid])[0]
pilot_sample_info = pl.read_excel(pilot_sample_info_xlsx, sheet_id = 2).head(8)

In [64]:
pilot_sample_info = pilot_sample_info.rename(
    {'__UNNAMED__0': 'row'}
).unpivot(
    index = 'row',
    variable_name = 'col',
    value_name = 'Sample'
).with_columns(
    pl.when(
        pl.col('col').str.len_chars() == 1
    ).then(pl.lit('0')
    ).otherwise(pl.lit('')
    ).alias('pad')
).with_columns(
    pl.concat_str(
        pl.col('row'),
        pl.col('pad'),
        pl.col('col')
    ).alias('Well')
).drop(['row','pad','col'])

In [65]:
pilot_sample_info.head()

Sample,Well
str,str
"""CAL 1""","""A01"""
"""CAL 2""","""B01"""
"""CAL 3""","""C01"""
"""CAL 4""","""D01"""
"""CAL 5""","""E01"""


### Read HAI experiments from HISE
Our first few batches of HAI have a slightly different format from other HAI results. We'll standardize data structure, filter data for relevant samples, and concatenate all values for downstream use.

In [66]:
hai_data_uuids = {
    'EXP-01042': '9c100dc9-60a7-4b98-90e7-69277b451c36',
    'EXP-01072': 'd0586978-716c-4df6-9b09-7d8d9f9d5a78',
    'EXP-01111': '3dd84f81-4392-4f66-996a-335975e9b40e',
    'PLAN-00144-1': 'abea7a06-ac2b-42d7-9002-2ff42a1310b6',
    'PLAN-00144-2': '6b34c7e6-1831-415c-9199-c438105060ce',
    'PLAN-00144-3': '32bee3f5-5162-4582-9768-2e1d8fdb3c7f',
    'PLAN-00144-4': 'a3ba8423-c172-4237-a825-a1a646342fd6',
    'PLAN-00144-5': 'f855b25a-9562-4134-ad43-b7d2edaff648',
    'PLAN-00144-6': 'f63d6eeb-0e34-4f34-9c21-01b68b562bcb',
    'PLAN-00144-7': '5f75b742-6385-4308-baf8-5c3c27a4f8a6'
}

### Prepare HAI data

Read and standardize the HAI experiment batches:

In [67]:
keep_cols = {
    'sample.sampleKitGuid': 'sample.sampleKitGuid',
    'Sample': 'specimen.specimenGuid',
    'Plate Name': 'msd.plateID', 
    'Well': 'msd.wellID', 
    'Assay': 'msd.assayName',
    'Signal': 'msd.signalWell',
    'Mean': 'msd.signalMean', 
    'CV': 'msd.signalPercentCV'
}

In [68]:
hai_list = []
for batch_id, uuid in hai_data_uuids.items():
    hai_csv = hisepy.cache_files([uuid])[0]
    hai_bn = os.path.basename(hai_csv)
    print(f'Reading {hai_bn}')

    # Initial batches have an extra header row that needs to be skipped
    if batch_id in ['EXP-01042', 'EXP-01072', 'EXP-01111']:
        n_skip = 1
    else:
        n_skip = 0
    
    # pandas here - for some reason, polars crashes the kernal with these
    all_hai = pd.read_csv(hai_csv, skiprows = n_skip)
    # but we can convert to polars for other steps
    all_hai = pl.DataFrame(all_hai)
    print(all_hai.shape)

    if 'Plate.Name' in all_hai.columns:
        all_hai = all_hai.rename({'Plate.Name': 'Plate Name', 'Sample Kit ID': 'sample.sampleKitGuid'})
    
    # Convert plate and sample name from the pilot batch
    if batch_id == 'EXP-01042':
        all_hai = all_hai.filter(
            pl.col('Plate Name') == 'Manual_1'
        ).drop('Sample').join(
            pilot_sample_info,
            how = 'left',
            on = 'Well'
        ).with_columns(
            pl.lit('EXP-01042_Plate_1').alias('Plate Name')
        )

    # convert specimens to sample kits
    all_hai = all_hai.with_columns(
        pl.Series(name = 'sample.sampleKitGuid',
                  values = specimens_to_kits(all_hai['Sample']))
    )
    # select and rename columns
    all_hai = all_hai.select(keep_cols.keys()).rename(keep_cols)
    
    # filter for samples in metadata
    sample_hai = all_hai.filter(
        pl.col('sample.sampleKitGuid').is_in(meta['sample.sampleKitGuid'].unique().to_list())
    )
    
    # add sample dilution value
    sample_hai = sample_hai.with_columns(
        pl.Series(name = 'msd.sampleDilution', values = [10000] * sample_hai.shape[0])
    )
    print('Sample data: {s}'.format(s = str(sample_hai.shape)))
    
    # filter for controls and standards
    control_hai = all_hai.filter(
        # Keep selected batches
        pl.col('msd.plateID').is_in(sample_hai['msd.plateID'].unique().to_list()),
        # Select first two columns
        pl.col('msd.wellID').is_in(control_samples.keys())
    ).drop(
        # Drop inconsistent values
        ['sample.sampleKitGuid','specimen.specimenGuid']
    )
    # apply corrected values for controls
    if batch_id in ['EXP-01042', 'EXP-01072']:
        # Pilot batches with 7 control dilutions
        control_hai = control_hai.with_columns(
            # Incorporate corrected values
            pl.Series(name = 'sample.sampleKitGuid', values = [pilot_samples[w] for w in control_hai['msd.wellID']]),
            pl.Series(name = 'specimen.specimenGuid', values = [pilot_specimens[w] for w in control_hai['msd.wellID']]),
            pl.Series(
                name = 'msd.sampleDilution', 
                values = [pilot_dilutions[w] for w in control_hai['msd.wellID']]
            )
        ).select(sample_hai.columns)
    else:
        # Later batches with 6 control dilutions
        control_hai = control_hai.with_columns(
            # Incorporate corrected values
            pl.Series(name = 'sample.sampleKitGuid', values = [control_samples[w] for w in control_hai['msd.wellID']]),
            pl.Series(name = 'specimen.specimenGuid', values = [control_specimens[w] for w in control_hai['msd.wellID']]),
            pl.Series(
                name = 'msd.sampleDilution', 
                values = [control_dilutions[w] for w in control_hai['msd.wellID']]
            )
        ).select(sample_hai.columns)
    print('Control data: {s}'.format(s = str(control_hai.shape)))

    # combine data and add batch ID
    hai_data = pl.concat([sample_hai, control_hai])
    hai_data = hai_data.with_columns(
        pl.lit(batch_id).alias('msd.batchID'),
        pl.col('msd.signalWell').cast(pl.Int64),
        pl.col('msd.signalMean').cast(pl.Float64)
    )
    print('Combined data: {s}'.format(s = str(hai_data.shape)))
    hai_list.append(hai_data)

Reading MSD_HAI_Pilot_DataTable.csv
(3840, 17)
Sample data: (800, 9)
Control data: (160, 9)
Combined data: (960, 10)
Reading EXP-01072 MSD Raw Data_pilot2.csv
(2880, 17)
Sample data: (2400, 9)
Control data: (480, 9)
Combined data: (2880, 10)
Reading EXP-01111 MSD HAI Data.csv
(2880, 17)
Sample data: (2160, 9)
Control data: (480, 9)
Combined data: (2640, 10)
Reading Plan-00144_MSD_HAI_Batch1_Datatable_for_HISE_ingest.csv
(2880, 13)
Sample data: (2200, 9)
Control data: (480, 9)
Combined data: (2680, 10)
Reading Plan-00144_MSD_HAI_Batch2_Datatable_for_HISE_ingest.csv
(2880, 13)
Sample data: (2360, 9)
Control data: (480, 9)
Combined data: (2840, 10)
Reading Plan-00144_MSD_HAI_Batch3_Datatable_for_HISE_ingest.csv
(2880, 13)
Sample data: (2300, 9)
Control data: (480, 9)
Combined data: (2780, 10)
Reading Plan-00144_MSD_HAI_Batch4_Datatable_for_HISE_ingest.csv
(2880, 13)
Sample data: (2260, 9)
Control data: (480, 9)
Combined data: (2740, 10)
Reading Plan-00144_MSD_HAI_Batch5_Datatable for HISE

In [69]:
all_hai = pl.concat(hai_list)

In [70]:
all_hai.shape

(24780, 10)

Are all samples available in the HAI data?

In [71]:
sum(meta['sample.sampleKitGuid'].is_in(sample_hai['sample.sampleKitGuid'].unique().to_list()))

79

### Compute Percent Inhibition per sample

In [72]:
sample_hai = all_hai.select(
    ['sample.sampleKitGuid', 'specimen.specimenGuid', 'msd.batchID', 'msd.plateID', 'msd.assayName']
).unique()
sample_hai.shape

(12390, 5)

#### Background normalization by BSA subtraction

Get BSA signal per well to use as background

In [73]:
bsa = all_hai.filter(
    pl.col('msd.assayName') == 'BSA'
).select(
    ['msd.plateID', 'msd.wellID', 'msd.signalWell']
).rename({'msd.signalWell': 'msd.signalBackground'})

Normalize by subtracting background

In [74]:
groups = ['sample.sampleKitGuid', 'specimen.specimenGuid', 'msd.plateID', 'msd.assayName']
norm_hai = all_hai.join(
    bsa,
    how = 'left',
    on = ['msd.plateID', 'msd.wellID']
).with_columns((
    pl.col('msd.signalWell') - pl.col('msd.signalBackground')
    ).alias('msd.signalNorm')
).with_columns(
    pl.when(pl.col('msd.signalNorm') < 0).then(0).otherwise(pl.col('msd.signalNorm')).alias('msd.signalNorm')
)

#### Mean and CV of normalized values

Compute means per sample

In [75]:
norm_means = norm_hai.group_by(*groups).agg(
    pl.mean('msd.signalNorm').alias('msd.signalNormMean')
)

Compute CV per sample

In [76]:
def pl_cv(x, col):
    return np.std(x[col].to_numpy()) / np.mean(x[col].to_numpy()) * 100

In [77]:
norm_cv = norm_hai.group_by(*groups).map_groups(
    lambda x: (
        x.with_columns(
            pl.Series(
                name = 'msd.signalNormPercentCV',
                values = [pl_cv(x, 'msd.signalNorm')] * x.shape[0]
            )
        )
    )
).select(
    ['sample.sampleKitGuid', 'specimen.specimenGuid', 'msd.plateID', 'msd.assayName', 'msd.signalNormPercentCV']
).unique()

/tmp/ipykernel_1397/3703753363.py:2: RuntimeWarning: invalid value encountered in scalar divide
  return np.std(x[col].to_numpy()) / np.mean(x[col].to_numpy()) * 100


Add Mean and CV to sample data

In [78]:
sample_hai = sample_hai.join(
    norm_means,
    how = 'left',
    on = groups
).join(
    norm_cv,
    how = 'left',
    on = groups
)

In [79]:
sample_hai.shape

(12390, 7)

In [80]:
sample_hai.head()

sample.sampleKitGuid,specimen.specimenGuid,msd.batchID,msd.plateID,msd.assayName,msd.signalNormMean,msd.signalNormPercentCV
str,str,str,str,str,f64,f64
"""KT00508""","""PL00508-06""","""PLAN-00144-2""","""2BMACAJ083_Batch2_Plate6""","""A/Wisconsin""",22748.0,2.712326
"""KT01593""","""PL01593-03""","""PLAN-00144-5""","""2BMACAD009_Batch5_Plate13""","""A/HongKong""",7847.0,13.712247
"""KT02259""","""PL02259-03""","""PLAN-00144-2""","""2BMACAJ083_Batch2_Plate6""","""A/Wisconsin""",18350.5,5.811831
"""KT02518""","""PL02518-003""","""PLAN-00144-5""","""2BMACAD009_Batch5_Plate13""","""B/Phuket""",432426.0,0.664391
"""HAI Reference Standard""","""HAI Reference Standard 1:16""","""PLAN-00144-6""","""Plate_2BMACAG057""","""A/HongKong""",5406.0,9.507954


#### Percent inhibition based on mean values

Compute inhibition based on blank controls from each plate - Diluent Only Control (Blank)

In [81]:
blanks = sample_hai.filter(
    pl.col('sample.sampleKitGuid') == 'Diluent Only Control (Blank)'
).select(
    ['msd.plateID', 'msd.assayName', 'msd.signalNormMean']
).rename({'msd.signalNormMean': 'msd.blankNormMean'}).unique()

In [82]:
sample_hai = sample_hai.join(
    blanks,
    how = 'left',
    on = ['msd.plateID', 'msd.assayName']
)

In [83]:
sample_hai.shape

(12390, 8)

In [84]:
sample_hai = sample_hai.with_columns(
    (
        (1 - pl.col('msd.signalNormMean')/pl.col('msd.blankNormMean')) * 100
    ).alias('msd.percentNormInhibition')
)

In [85]:
sample_hai.head()

sample.sampleKitGuid,specimen.specimenGuid,msd.batchID,msd.plateID,msd.assayName,msd.signalNormMean,msd.signalNormPercentCV,msd.blankNormMean,msd.percentNormInhibition
str,str,str,str,str,f64,f64,f64,f64
"""KT00508""","""PL00508-06""","""PLAN-00144-2""","""2BMACAJ083_Batch2_Plate6""","""A/Wisconsin""",22748.0,2.712326,30884.0,26.343738
"""KT01593""","""PL01593-03""","""PLAN-00144-5""","""2BMACAD009_Batch5_Plate13""","""A/HongKong""",7847.0,13.712247,7501.0,-4.612718
"""KT02259""","""PL02259-03""","""PLAN-00144-2""","""2BMACAJ083_Batch2_Plate6""","""A/Wisconsin""",18350.5,5.811831,30884.0,40.582502
"""KT02518""","""PL02518-003""","""PLAN-00144-5""","""2BMACAD009_Batch5_Plate13""","""B/Phuket""",432426.0,0.664391,828841.0,47.827629
"""HAI Reference Standard""","""HAI Reference Standard 1:16""","""PLAN-00144-6""","""Plate_2BMACAG057""","""A/HongKong""",5406.0,9.507954,9281.0,41.751966


### Retrospectively join sample results to the raw data

In [86]:
raw_hai = norm_hai.join(
    sample_hai,
    how = 'left',
    on = ['sample.sampleKitGuid', 'specimen.specimenGuid', 'msd.batchID', 'msd.plateID', 'msd.assayName']
)

In [87]:
raw_hai.shape

(24780, 16)

### Join assay and sample metadata

In [88]:
raw_hai = raw_hai.join(
    meta,
    how = 'left',
    on = 'sample.sampleKitGuid'
).join(
    assay_meta,
    how = 'left',
    on = 'msd.assayName'
)

In [89]:
hai_order = [
    'cohort.cohortGuid',
    'subject.subjectGuid', 'subject.birthYear', 'subject.ageAtFirstDraw', 'subject.ageGroup', 
    'subject.biologicalSex', 'subject.race', 'subject.ethnicity', 'subject.cmv',
    'sample.sampleKitGuid', 'sample.visitName', 'sample.drawYear', 'sample.subjectAgeAtDraw',
    'sample.daysSinceFirstVisit', 'vaccine.year',
    'specimen.specimenGuid',
    'msd.batchID', 'msd.plateID', 'msd.wellID',
    'msd.antigenName', 'msd.antigenSubtype', 'msd.antigenFullName',
    'msd.sampleDilution',
    'msd.signalWell', 'msd.signalBackground', 'msd.signalNorm', 
    'msd.signalNormMean', 'msd.signalNormPercentCV', 'msd.blankNormMean',
    'msd.percentNormInhibition'
]

In [90]:
raw_hai = raw_hai.select(hai_order)

In [91]:
raw_hai.shape

(24780, 30)

### Separate duplicate measures

Some samples have been measured more than once across multiple batches. We'll separate out these duplicate measures, as they can be confusing for downstream analysis. 

**Single measures** retain the most recent measurement in these cases by sorting on batch and plate IDs (which are alphanumerically ordered).

**Duplicate measures** retain all measurements of the duplicated samples, including the most recent, to enable analysis of cross-batch replication.

In [92]:
treat_hai = raw_hai.filter(
    pl.col('specimen.specimenGuid').str.contains('PL')
)
control_hai = raw_hai.filter(
    ~pl.col('specimen.specimenGuid').str.contains('PL')
)

In [93]:
last_measures = treat_hai.select(
    ['sample.sampleKitGuid','specimen.specimenGuid', 'msd.batchID', 'msd.plateID']
).unique(    
).sort(
    ['sample.sampleKitGuid', 'specimen.specimenGuid', 'msd.batchID', 'msd.plateID']
).group_by('sample.sampleKitGuid').last()

In [94]:
single_hai = treat_hai.filter(
    pl.col('specimen.specimenGuid').is_in(last_measures['specimen.specimenGuid'].to_list())
)

duplicate_kits = treat_hai.filter(
    ~pl.col('specimen.specimenGuid').is_in(last_measures['specimen.specimenGuid'].to_list())
)['sample.sampleKitGuid'].to_list()
duplicate_hai = treat_hai.filter(
    ~pl.col('sample.sampleKitGuid').is_in(duplicate_kits)
)

one plate is entirely duplicated

In [95]:
dup_plate = list(set(treat_hai['msd.plateID'].unique()) - set(single_hai['msd.plateID'].unique()))
dup_plate

['EXP-01042_Plate_1']

In [96]:
single_control_hai = control_hai.filter(
    pl.col('msd.plateID').is_in(single_hai['msd.plateID'].unique().to_list())
)
duplicate_control_hai = control_hai.filter(
    pl.col('msd.plateID').is_in(duplicate_hai['msd.plateID'].unique().to_list())
)

In [97]:
single_hai = pl.concat(
    [single_hai, single_control_hai]
)
duplicate_hai = pl.concat(
    [duplicate_hai, duplicate_control_hai]
)

In [98]:
single_hai.shape

(22600, 30)

In [99]:
duplicate_hai.shape

(20200, 30)

### Assemble sample-level data

For sample-level results, we'll exclude controls to provide the simplest data to end users looking for on-study results

In [100]:
single_sample_hai = single_hai.drop(
    ['msd.wellID','msd.signalWell','msd.signalBackground','msd.signalNorm']
).filter(
    pl.col('specimen.specimenGuid').str.contains('PL')
).unique()
single_sample_hai.shape

(9140, 26)

In [101]:
duplicate_sample_hai = duplicate_hai.drop(
    ['msd.wellID','msd.signalWell','msd.signalBackground','msd.signalNorm']
).filter(
    pl.col('specimen.specimenGuid').str.contains('PL')
).unique()
duplicate_sample_hai.shape

(8180, 26)

In [102]:
single_sample_hai['sample.visitName'].value_counts()

sample.visitName,count
str,u32
"""Flu Year 2 Stand-Alone""",220
"""Flu Year 3 Stand-Alone""",470
"""Flu Year 1 Day 0""",1080
"""Flu Year 1 Day 90""",1030
"""Flu Year 2 Day 90""",820
…,…
"""Immune Variation Day 90""",840
"""Immune Variation Day 0""",890
"""Flu Year 2 Day 0""",840


In [103]:
single_sample_serology['sample.visitName'].value_counts()

sample.visitName,count
str,u32
"""Flu Year 2 Day 7""",589
"""Flu Year 2 Day 0""",589
"""Flu Year 2 Day 90""",574
"""Flu Year 1 Day 90""",623
"""Flu Year 1 Day 0""",647
"""Flu Year 1 Day 7""",645


### Output HAI results

In [104]:
single_hai_out = 'output/sound-life_flu_hai_single_raw_{d}.csv'.format(d = date.today())
single_hai.write_csv(single_hai_out)
out_files.append(single_hai_out)

In [105]:
duplicate_hai_out = 'output/sound-life_flu_hai_duplicates_raw_{d}.csv'.format(d = date.today())
duplicate_hai.write_csv(duplicate_hai_out)
out_files.append(duplicate_hai_out)

In [106]:
single_sample_hai_out = 'output/sound-life_flu_hai_single_{d}.csv'.format(d = date.today())
single_sample_hai.write_csv(single_sample_hai_out)
out_files.append(single_sample_hai_out)

In [107]:
duplicate_sample_hai_out = 'output/sound-life_flu_hai_duplicates_{d}.csv'.format(d = date.today())
duplicate_sample_hai.write_csv(duplicate_sample_hai_out)
out_files.append(duplicate_sample_hai_out)

## Upload data to HISE

Finally, we'll use `hisepy.upload.upload_files()` to send a copy of our output to HISE to use for distribution.

In [108]:
study_space_uuid = 'de025812-5e73-4b3c-9c3b-6d0eac412f2a'
title = 'DIHA serology and HAI data {d}'.format(d = date.today())

In [109]:
search_id = element_id()
search_id

'iodine-tennessine-berkelium'

In [110]:
in_files = [assay_meta_uuid, meta_uuid, std_uuid, msd_uuid] + list(hai_data_uuids.values())
in_files

['5e3a7f65-d92a-4af0-8983-1d31a9000e14',
 'af25e3e7-25c1-4476-afb4-926bd201db8f',
 '878ed657-7c57-4b6b-8d2b-da68c243da88',
 'df68e382-a14e-4d86-b5c8-25963c084a54',
 '9c100dc9-60a7-4b98-90e7-69277b451c36',
 'd0586978-716c-4df6-9b09-7d8d9f9d5a78',
 '3dd84f81-4392-4f66-996a-335975e9b40e',
 'abea7a06-ac2b-42d7-9002-2ff42a1310b6',
 '6b34c7e6-1831-415c-9199-c438105060ce',
 '32bee3f5-5162-4582-9768-2e1d8fdb3c7f',
 'a3ba8423-c172-4237-a825-a1a646342fd6',
 'f855b25a-9562-4134-ad43-b7d2edaff648',
 'f63d6eeb-0e34-4f34-9c21-01b68b562bcb',
 '5f75b742-6385-4308-baf8-5c3c27a4f8a6']

In [111]:
out_files

['output/sound-life_flu_serology_single_raw_2025-08-07.csv',
 'output/sound-life_flu_serology_duplicates_raw_2025-08-07.csv',
 'output/sound-life_flu_serology_single_2025-08-07.csv',
 'output/sound-life_flu_serology_duplicates_2025-08-07.csv',
 'output/sound-life_flu_hai_single_raw_2025-08-07.csv',
 'output/sound-life_flu_hai_duplicates_raw_2025-08-07.csv',
 'output/sound-life_flu_hai_single_2025-08-07.csv',
 'output/sound-life_flu_hai_duplicates_2025-08-07.csv']

In [112]:
len(out_files)

8

In [113]:
hisepy.upload.upload_files(
    files = out_files,
    study_space_id = study_space_uuid,
    title = title,
    input_file_ids = in_files,
    destination = search_id
)

checking if conda environment can compile...


{'Message': 'General Okay-ness',
 'VisualizationId': '00000000-0000-0000-0000-000000000000',
 'AbstractionId': '00000000-0000-0000-0000-000000000000',
 'TraceId': '5a6b66f7-fb32-44f8-a35a-13e1bdb28030',
 'ProcessId': 'b36d2766-8b3d-4c90-a3ad-3df074442b4c',
 'WorkflowId': '7005806e-c185-456b-bf9b-8fa4d2e9b5e2',
 'FileIds': ['e5030014-b050-4cc6-9736-fabe6e860084',
  '5a95e07c-fe81-41e8-b48d-18ffbdc7ab52',
  '69c3a7f5-45d7-4bc9-9e78-79eba6cf0f1d',
  'e99cce34-ad45-4ed7-b1a0-fb9d18d0385e',
  '99e16509-4388-4cba-9d81-e504a9f4ada6',
  '54f64ec1-4be7-4e1c-8f4e-2f154468feb1',
  '45f9d911-f018-461b-8a8a-8e724ae2ca2c',
  '3b455305-a804-421e-9be3-b2bcfef028e0']}

In [114]:
import session_info
session_info.show()